# ArmorVault — Synthetic OCR Benchmark (CPU v2)

CPU-only. Creates 12 synthetic documents, runs PP-OCRv5, and prints field-level accuracy without E5.

In [ ]:
%pip install -q "rapidocr>=3.9,<4" "onnxruntime>=1.20" huggingface_hub "pyyaml>=6" "python-bidi>=0.6,<1" "arabic-reshaper>=3,<4" pillow

In [ ]:
import json, shutil, subprocess, sys
from pathlib import Path
from urllib.request import urlretrieve

repo = Path('/content/armorvault-ocr-vl-gpu-lab')
dataset = Path('/content/armorvault-synthetic-benchmark')
result_file = Path('/content/armorvault-synthetic-results.json')
shutil.rmtree(repo, ignore_errors=True)
shutil.rmtree(dataset, ignore_errors=True)
result_file.unlink(missing_ok=True)

def run(command):
    print('+', ' '.join(map(str, command)), flush=True)
    subprocess.run(command, check=True)

run(['git', 'clone', '-q', 'https://github.com/almawti/armorvault-ocr-vl-gpu-lab.git', str(repo)])
font = Path('/content/NotoSansArabic.ttf')
urlretrieve('https://raw.githubusercontent.com/google/fonts/main/ofl/notosansarabic/NotoSansArabic%5Bwdth,wght%5D.ttf', font)
if not font.exists() or font.stat().st_size < 10000:
    raise RuntimeError('Arabic font download failed')

run([sys.executable, str(repo / 'generate_synthetic_ocr_benchmark.py'), '--output', str(dataset), '--font', str(font)])
run([sys.executable, str(repo / 'run_synthetic_ocr_benchmark.py'), '--dataset', str(dataset), '--output', str(result_file)])
if not result_file.exists():
    raise RuntimeError('Benchmark process ended without creating its result file')

result = json.loads(result_file.read_text(encoding='utf-8'))
summary = {key: value for key, value in result.items() if key != 'reports'}
print('\nFINAL RESULT')
print(json.dumps(summary, ensure_ascii=False, indent=2))